In [4]:
#########################################################################
########------ Ciencia de Datos e IA Generativa con Python ------########
#########################################################################
# Capacitador: Julio César Bernal Fernández
# email: juliobf08@gmail.com
# Tema : Caso de Uso de Sistemas de Recomendación Peliculas
# versión: 1.0
#########################################################################

#🎬 **Caso real: Sistema de Recomendación con MovieLens 1M**

Dataset: MovieLens 1M (1 millón de calificaciones de usuarios sobre películas).
Se descarga directamente con wget o requests, sin credenciales.

In [2]:
# ==========================================
# A) IMPORTS Y UTILIDADES (sin compilaciones)
#    - Esta celda NO instala nada; sólo prepara el entorno.
# ==========================================
import os                 # Operaciones del sistema (paths, proceso)
import math               # Funciones matemáticas básicas
import random             # Semillas para reproducibilidad
from pathlib import Path  # Manejo robusto de rutas

import numpy as np        # Cálculo numérico
import pandas as pd       # Manejo de DataFrames (tablas)

# Creamos un generador de números aleatorios con semilla fija (reproducible)
rng = np.random.default_rng(42)

def set_seed(seed=42):
    # Fija semillas para random (Python) y NumPy
    random.seed(seed)
    np.random.seed(seed)

# Fijamos semillas al arrancar
set_seed(42)
print("✅ Entorno listo (puro NumPy/pandas, sin compilar)")


✅ Entorno listo (puro NumPy/pandas, sin compilar)


# Cargar MovieLens 1M

In [3]:
# ==========================================
# B) DATOS: MOVIELENS 1M DESDE /content
#    - Asegúrate de haber subido /content/ml-1m.zip
#    - Descomprime en /content/data/ml-1m
#    - Lee users.dat, movies.dat, ratings.dat (separador '::')
# ==========================================
import zipfile             # Descomprimir ZIP

In [4]:
# Ruta al ZIP (ajusta el nombre si lo subiste distinto)
ZIP_PATH = Path("/content/ml-1m.zip")

# Carpeta donde extraeremos el ZIP
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# Validamos que el ZIP exista
assert ZIP_PATH.exists(), f"No se encontró {ZIP_PATH}. Sube 'ml-1m.zip' a /content."

In [6]:
# Definimos carpeta esperada que crea el ZIP al extraerse
ML1M_DIR = DATA_DIR / "ml-1m"

# Si aún no está descomprimido, lo extraemos
if not ML1M_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:  # Abrimos ZIP
        zf.extractall(DATA_DIR)                 # Extraemos a DATA_DIR

In [7]:
# Validamos que la carpeta 'ml-1m' exista
assert ML1M_DIR.exists(), f"No se encontró carpeta 'ml-1m' dentro de {DATA_DIR}."

In [8]:
# Rutas a archivos .dat
users_path   = ML1M_DIR / "users.dat"
movies_path  = ML1M_DIR / "movies.dat"
ratings_path = ML1M_DIR / "ratings.dat"

In [9]:
# Leemos cada .dat con separador '::' y encoding latin-1 (evita problemas de acentos)
users = pd.read_csv(
    users_path, sep="::", header=None, engine="python", encoding="latin-1",
    names=["user_id", "gender", "age", "occupation", "zip"]
)
movies = pd.read_csv(
    movies_path, sep="::", header=None, engine="python", encoding="latin-1",
    names=["movie_id", "title", "genres"]
)
ratings = pd.read_csv(
    ratings_path, sep="::", header=None, engine="python", encoding="latin-1",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

In [10]:
# Tipos adecuados
ratings["user_id"]  = ratings["user_id"].astype(int)
ratings["movie_id"] = ratings["movie_id"].astype(int)
ratings["rating"]   = ratings["rating"].astype(float)

In [11]:
# Vistazo rápido
print(f"users:   {users.shape}")
print(f"movies:  {movies.shape}")
print(f"ratings: {ratings.shape}")
display(ratings.head(3))
display(movies.head(3))

users:   (6040, 5)
movies:  (3883, 3)
ratings: (1000209, 4)


,user_id,movie_id,rating,timestamp
0,1,1193,5.0,978300760
1,1,661,3.0,978302109
2,1,914,3.0,978301968


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance


In [ ]:
# ==========================================
# C) PREPROCESO: MAPEOS, SPLIT 80/20 Y MATRICES
#    - Reindexa user_id y movie_id a índices 0..U-1 y 0..I-1
#    - Split aleatorio 80/20 para train/test
# ==========================================

# Reindexado, split 80/20, preparación

In [12]:
set_seed(42)  # Reafirmamos semillas

# IDs únicos
unique_users = ratings["user_id"].unique()
unique_items = ratings["movie_id"].unique()

# Mapas id_real -> índice interno
uid2idx = {u:i for i,u in enumerate(unique_users)}
iid2idx = {m:i for i,m in enumerate(unique_items)}

# Mapas inversos índice interno -> id_real (útil para mostrar)
idx2uid = {i:u for u,i in uid2idx.items()}
idx2iid = {i:m for m,i in iid2idx.items()}

# Copia con columnas de índices internos 'u' y 'i'
ratings_idx = ratings.copy()
ratings_idx["u"] = ratings_idx["user_id"].map(uid2idx)
ratings_idx["i"] = ratings_idx["movie_id"].map(iid2idx)

# Split aleatorio 80/20 a nivel de filas (simple y suficiente para demo)
mask = rng.random(len(ratings_idx)) < 0.8
train_df = ratings_idx[mask].reset_index(drop=True)
test_df  = ratings_idx[~mask].reset_index(drop=True)

# Stats y media global
global_mean = train_df["rating"].mean()
n_users = len(unique_users)
n_items = len(unique_items)
print(f"Train/Test: {train_df.shape} / {test_df.shape}")
print(f"U={n_users} | I={n_items} | global_mean={global_mean:.4f}")


Train/Test: (800015, 6) / (200194, 6)
U=6040 | I=3706 | global_mean=3.5819


# FunkSVD (SGD puro NumPy) + entrenamiento

In [13]:
# ==========================================
# D) FUNKSVD (SGD) PURO NUMPY
#    - Aprende P (usuarios), Q (ítems), sesgos bu/bi y media global μ
#    - Predicción: r_hat = μ + bu + bi + P_u · Q_i
#    - Para clase: por defecto entrena con muestra de 200k ratings (acelera)
# ==========================================

def train_funksvd(
    df, n_users, n_items,
    n_factors=64, n_epochs=10,
    lr=0.01, reg=0.02,
    use_bias=True, sample_size=200_000,  # ⇦ Pon None para usar TODO el train (más lento)
    seed=42
):
    # RNG local para inicializaciones y muestreos
    rng_local = np.random.default_rng(seed)
    # Inicializaciones pequeñas (evita estallidos al inicio)
    P = 0.01 * rng_local.standard_normal((n_users, n_factors))  # Embeddings de usuarios
    Q = 0.01 * rng_local.standard_normal((n_items, n_factors))  # Embeddings de ítems
    bu = np.zeros(n_users)                                      # Sesgos de usuario
    bi = np.zeros(n_items)                                      # Sesgos de ítem

    # Media global sobre el set de entrenamiento
    mu = df["rating"].mean()

    # Pasamos a arrays para loop rápido
    u_arr = df["u"].to_numpy()
    i_arr = df["i"].to_numpy()
    r_arr = df["rating"].to_numpy()

    n_samples = len(df)
    idx = np.arange(n_samples)

    # Muestreamos para acelerar (suficiente para demo en clase)
    if sample_size is not None and sample_size < n_samples:
        sel = rng_local.choice(n_samples, size=sample_size, replace=False)
        u_arr = u_arr[sel]
        i_arr = i_arr[sel]
        r_arr = r_arr[sel]
        idx = np.arange(len(sel))

    # SGD: recorre aleatoriamente las observaciones y actualiza parámetros
    for epoch in range(1, n_epochs+1):
        rng_local.shuffle(idx)
        for t in idx:
            u = u_arr[t]             # índice de usuario
            i = i_arr[t]             # índice de ítem
            r = r_arr[t]             # rating real

            # Predicción actual
            pred = mu + (bu[u] + bi[i] if use_bias else 0.0) + P[u].dot(Q[i])

            # Error residual
            e = r - pred

            # Guardamos copias de P[u] y Q[i] antes de actualizar (para usar en ambos updates)
            Pu = P[u].copy()
            Qi = Q[i].copy()

            # Actualizamos sesgos si están activados
            if use_bias:
                bu[u] += lr * (e - reg * bu[u])
                bi[i] += lr * (e - reg * bi[i])

            # Actualizamos factores latentes (con regularización L2)
            P[u] += lr * (e * Qi - reg * Pu)
            Q[i] += lr * (e * Pu - reg * Qi)

        # Feedback cada 2 épocas
        if epoch % 2 == 0 or epoch == n_epochs:
            print(f"Época {epoch}/{n_epochs} completada.")

    # Devolvemos el "modelo" como dict simple
    return {"P": P, "Q": Q, "bu": bu, "bi": bi, "mu": mu, "use_bias": use_bias}

def predict_single(model, u, i):
    # Recuperamos parámetros
    P, Q, bu, bi, mu, use_bias = model["P"], model["Q"], model["bu"], model["bi"], model["mu"], model["use_bias"]
    # Predicción con (o sin) sesgos
    est = mu + (bu[u] + bi[i] if use_bias else 0.0) + P[u].dot(Q[i])
    # Aseguramos que esté en rango [1,5] como MovieLens
    return float(np.clip(est, 1.0, 5.0))

# Entrenamos el modelo con hiperparámetros razonables para demo
model = train_funksvd(
    train_df, n_users, n_items,
    n_factors=100,   # dimensión de embeddings
    n_epochs=12,     # épocas de entrenamiento
    lr=0.01, reg=0.02,
    use_bias=True,
    sample_size=200_000,  # ⇦ subir a None para usar todo el train (irá más lento)
    seed=42
)
print("✅ Modelo entrenado (FunkSVD)")


Época 2/12 completada.
Época 4/12 completada.
Época 6/12 completada.
Época 8/12 completada.
Época 10/12 completada.
Época 12/12 completada.
✅ Modelo entrenado (FunkSVD)


# Evaluación (RMSE, MAE) en Test

In [14]:
# ==========================================
# E) EVALUACIÓN: RMSE y MAE en TEST
#    - Calcula errores vs ratings reales del test
# ==========================================
from math import sqrt  # Raíz cuadrada para RMSE

def eval_rmse_mae(model, df):
    # Listas para predicciones y errores
    preds = []
    errs  = []
    # Recorremos cada fila del set de test
    for row in df.itertuples(index=False):
        u, i, r = row.u, row.i, row.rating              # índices y rating real
        est = predict_single(model, u, i)               # predicción del modelo
        preds.append(est)                               # guardamos predicción
        errs.append(r - est)                            # error residual
    # Convertimos a arrays NumPy
    preds = np.array(preds, dtype=np.float64)
    errs  = np.array(errs,  dtype=np.float64)
    # RMSE y MAE
    rmse = sqrt(np.mean(errs**2))
    mae  = float(np.mean(np.abs(errs)))
    return rmse, mae

# Medimos en el test
rmse, mae = eval_rmse_mae(model, test_df)
print(f"✅ Test | RMSE={rmse:.4f} | MAE={mae:.4f}")


✅ Test | RMSE=0.9224 | MAE=0.7290


# Top-N recomendaciones para un usuario (evitando vistos)

In [15]:
# ==========================================
# F) TOP-N RECOMENDACIONES PARA UN USUARIO
#    - Evita recomendar ítems ya vistos por el usuario en train
#    - Devuelve lista (movie_id real, título, score estimado)
# ==========================================
movieid_to_title = dict(zip(movies["movie_id"], movies["title"]))  # Mapa id->título

def get_seen_set(df, user_idx):
    # Conjunto de ítems que el usuario ya calificó en TRAIN
    return set(df.loc[df.u == user_idx, "i"].tolist())

def recommend_top_n(model, user_idx, N=10):
    # Ítems ya vistos por el usuario (no recomendarlos)
    seen = get_seen_set(train_df, user_idx)
    # Candidatos = todos los ítems que NO están en 'seen'
    candidates = np.fromiter((i for i in range(n_items) if i not in seen), dtype=np.int32)
    # Estimamos score para cada candidato
    ests = []
    for i in candidates:
        ests.append(predict_single(model, user_idx, i))
    ests = np.array(ests)
    # Ordenamos de mayor a menor y tomamos Top-N
    order = np.argsort(ests)[::-1][:N]
    top_i = candidates[order]
    top_scores = ests[order]
    # Convertimos índices internos a IDs reales y títulos
    top_movie_ids = [idx2iid[i] for i in top_i]
    top_titles = [movieid_to_title.get(mid, f"Movie {mid}") for mid in top_movie_ids]
    # Empaquetamos
    return list(zip(top_movie_ids, top_titles, top_scores))

# Elegimos un usuario de ejemplo (índice interno)
user_idx_example = int(train_df["u"].sample(1, random_state=7).iloc[0])

# Obtenemos Top-10
top10 = recommend_top_n(model, user_idx_example, N=10)
print(f"👤 Usuario (índice interno): {user_idx_example} | user_id real: {idx2uid[user_idx_example]}")
print("🎬 TOP-10 Recomendaciones:")
for mid, title, score in top10:
    print(f"  - {title}  (est={score:.3f})")


👤 Usuario (índice interno): 2566 | user_id real: 2567
🎬 TOP-10 Recomendaciones:
  - Schindler's List (1993)  (est=4.310)
  - Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)  (est=4.295)
  - Wrong Trousers, The (1993)  (est=4.289)
  - Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)  (est=4.283)
  - Rear Window (1954)  (est=4.267)
  - Close Shave, A (1995)  (est=4.263)
  - Third Man, The (1949)  (est=4.233)
  - To Kill a Mockingbird (1962)  (est=4.223)
  - Creature Comforts (1990)  (est=4.211)
  - General, The (1927)  (est=4.207)


# Precision@K y Recall@K (umbral de relevancia)

In [16]:
# ==========================================
# G) MÉTRICAS TOP-K (Precision@K, Recall@K)
#    - Relevante: rating real >= 4.0
#    - Evaluamos por usuario en su porción del test
# ==========================================
from collections import defaultdict  # Agrupar ítems por usuario

K = 10           # Tamaño del Top-K
THRESH = 4.0     # Umbral de relevancia

# Agrupamos test por usuario: u -> [(i, rating), ...]
per_user = defaultdict(list)
for row in test_df.itertuples(index=False):
    per_user[row.u].append((row.i, row.rating))

precisions, recalls = [], []

for u, items in per_user.items():
    # Para cada ítem del usuario, calculamos score estimado
    scored = [(i, r, predict_single(model, u, i)) for (i, r) in items]
    # Ordenamos por score desc
    scored.sort(key=lambda x: x[2], reverse=True)
    # Top-K
    topk = scored[:K]
    # Conteos de relevantes en Top-K y total relevantes del usuario
    rel_in_topk = sum(1 for (_, r, _) in topk if r >= THRESH)
    total_rel   = sum(1 for (_, r, _) in scored if r >= THRESH)
    # Precisión y recall por usuario (evitando div/0)
    prec = rel_in_topk / K if K > 0 else 0.0
    rec  = (rel_in_topk / total_rel) if total_rel > 0 else 0.0
    # Acumulamos
    precisions.append(prec)
    recalls.append(rec)

# Promedios por usuario
P_at_K = float(np.mean(precisions)) if precisions else 0.0
R_at_K = float(np.mean(recalls)) if recalls else 0.0
print(f"📐 Top-{K} | Precision@K={P_at_K:.4f} | Recall@K={R_at_K:.4f} (relevancia ≥ {THRESH})")


📐 Top-10 | Precision@K=0.6653 | Recall@K=0.6273 (relevancia ≥ 4.0)


# “Favoritos” vs recomendaciones (vista para demo en clase)

In [17]:
# ==========================================
# H) VISTA PARA CLASE: FAVORITOS VS RECOMENDACIONES
#    - Muestra los 'favoritos' (ratings altos) de un usuario
#    - Muestra sus Top-N estimados por el modelo
# ==========================================
def user_top_likes(df_ratings_original, user_id_real, min_rating=4.5, n=10):
    # Filtra ratings del usuario con puntuación alta, ordena desc y pega títulos
    favs = (df_ratings_original[df_ratings_original.user_id == user_id_real]
            .query("rating >= @min_rating")
            .sort_values("rating", ascending=False)
            .head(n))
    return favs.merge(movies[["movie_id","title"]], on="movie_id", how="left")[["movie_id","title","rating"]]

# user_id REAL asociado al user_idx_example
uid_real = idx2uid[user_idx_example]
print(f"👤 user_id real: {uid_real} — favoritos (rating ≥ 4.5):")
display(user_top_likes(ratings, uid_real, min_rating=4.5, n=10))

print("🎁 Recomendaciones Top-10 (FunkSVD):")
pd.DataFrame(recommend_top_n(model, user_idx_example, N=10), columns=["movie_id","title","est_rating"])


👤 user_id real: 2567 — favoritos (rating ≥ 4.5):


,movie_id,title,rating
0,1249,Nikita (La Femme Nikita) (1990),5.0
1,1250,"Bridge on the River Kwai, The (1957)",5.0
2,1266,Unforgiven (1992),5.0
3,750,Dr. Strangelove or: How I Learned to Stop Worr...,5.0
4,3035,Mister Roberts (1955),5.0
5,923,Citizen Kane (1941),5.0
6,924,2001: A Space Odyssey (1968),5.0
7,3062,"Longest Day, The (1962)",5.0
8,3091,Kagemusha (1980),5.0
9,1653,Gattaca (1997),5.0


🎁 Recomendaciones Top-10 (FunkSVD):


,movie_id,title,est_rating
0,527,Schindler's List (1993),4.310297
1,2019,Seven Samurai (The Magnificent Seven) (Shichin...,4.294803
2,1148,"Wrong Trousers, The (1993)",4.288975
3,922,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),4.282683
4,904,Rear Window (1954),4.267028
5,745,"Close Shave, A (1995)",4.263152
6,1212,"Third Man, The (1949)",4.233018
7,1207,To Kill a Mockingbird (1962),4.223496
8,3429,Creature Comforts (1990),4.210843
9,3022,"General, The (1927)",4.207318
